# Stage 02 & 03: Exploratory Data Analysis & Business SQL Analysis
**Project**: Steam Game Intelligence  
**Notebook**: `notebooks/02_eda_and_business_analysis.ipynb`  
**Objective**: Execute exploratory statistical visualizations (catalog genre breakdown, player rating distribution, review engagement vs recommendation rate) and reproduce business SQL queries using DuckDB.

---
## Business Questions Addressed:
1. **Genre Dominance**: Which genres dominate catalog volume, review engagement, and player reception?
2. **Rank Divergence**: Which games exhibit significant divergence between Sales Rank and Review Rank (e.g. commercial hits with low player satisfaction vs hidden gems)?
3. **Monetization Strength**: How does Revenue Rank compare against Sales Rank across base games and DLCs?
4. **Engagement Patterns**: How does median player playtime correlate with recommendation rate and review helpfulness?


In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.size'] = 11

con = duckdb.connect(database=':memory:')
desc_path = '../data/processed/games_description_clean.csv' if os.path.exists('../data/processed/games_description_clean.csv') else 'data/processed/games_description_clean.csv'
rank_path = '../data/processed/games_ranking_clean.csv' if os.path.exists('../data/processed/games_ranking_clean.csv') else 'data/processed/games_ranking_clean.csv'
rev_path = '../data/processed/steam_game_reviews_clean.csv' if os.path.exists('../data/processed/steam_game_reviews_clean.csv') else 'data/processed/steam_game_reviews_clean.csv'

con.execute(f"CREATE TABLE games_desc AS SELECT * FROM read_csv_auto('{desc_path}')")
con.execute(f"CREATE TABLE games_rank AS SELECT * FROM read_csv_auto('{rank_path}')")
con.execute(f"CREATE TABLE steam_reviews AS SELECT * FROM read_csv_auto('{rev_path}')")

print("DuckDB Tables Loaded Successfully.")


### 1. Catalog & Genre Distribution Analysis
We analyze genre frequency and review volume distribution across Steam titles.


In [ ]:
df_genre = con.execute("""
    WITH genre_split AS (
        SELECT trim(replace(replace(replace(g.genre, '[', ''), ']', ''), '''', '')) AS genre, 
               d.name, d.number_of_reviews_from_purchased_people_clean
        FROM games_desc d, UNNEST(string_split(d.genres, ',')) AS g(genre)
    )
    SELECT genre, COUNT(DISTINCT name) AS total_games, SUM(number_of_reviews_from_purchased_people_clean) AS total_reviews
    FROM genre_split
    WHERE genre != ''
    GROUP BY genre
    ORDER BY total_games DESC
""").df()

plt.figure(figsize=(10, 5))
sns.barplot(data=df_genre.head(10), x='total_games', y='genre', hue='genre', legend=False, palette='Blues_r')
plt.title('Top 10 Steam Catalog Genres by Game Count')
plt.xlabel('Number of Games')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

df_genre.head(10)


### 2. Commercial Rank vs Review Rank Divergence (SQL Query 2)
Investigating games where Sales Rank is significantly higher or lower than Review Rank.


In [ ]:
df_divergence = con.execute("""
    WITH rank_pivoted AS (
        SELECT game_name, normalized_game_name, title_classification,
            MAX(CASE WHEN rank_type = 'Revenue' THEN rank_clean END) AS revenue_rank,
            MAX(CASE WHEN rank_type = 'Sales' THEN rank_clean END) AS sales_rank,
            MAX(CASE WHEN rank_type = 'Review' THEN rank_clean END) AS review_rank
        FROM games_rank
        GROUP BY game_name, normalized_game_name, title_classification
    )
    SELECT game_name, sales_rank, review_rank, (sales_rank - review_rank) AS sales_minus_review_diff,
        CASE 
            WHEN (sales_rank - review_rank) < -30 THEN 'High Review / Low Sales (Hidden Gem)'
            WHEN (sales_rank - review_rank) > 30 THEN 'High Sales / Low Review (Commercial Success w/ Friction)'
            ELSE 'Aligned Rank'
        END AS category
    FROM rank_pivoted
    WHERE sales_rank IS NOT NULL AND review_rank IS NOT NULL
    ORDER BY ABS(sales_rank - review_rank) DESC
    LIMIT 20
""").df()

plt.figure(figsize=(10, 5))
sns.scatterplot(data=df_divergence, x='sales_rank', y='review_rank', hue='category', s=100)
plt.title('Sales Rank vs Review Rank Scatter (Divergence Analysis)')
plt.xlabel('Sales Rank (Lower is Better)')
plt.ylabel('Review Rank (Lower is Better)')
plt.tight_layout()
plt.show()

df_divergence


### 3. Review Engagement & Playtime vs Recommendation Rate
Analyzing user review micro-data (~992k reviews) to test the relationship between median hours played and recommendation rate.


In [ ]:
df_engagement = con.execute("""
    SELECT 
        game_name,
        COUNT(*) AS review_count,
        ROUND(MEDIAN(hours_played_clean), 1) AS median_playtime_hours,
        ROUND(AVG(is_recommended) * 100, 2) AS recommendation_pct
    FROM steam_reviews
    GROUP BY game_name
    HAVING COUNT(*) >= 1000
    ORDER BY review_count DESC
    LIMIT 25
""").df()

plt.figure(figsize=(9, 5))
sns.regplot(data=df_engagement, x='median_playtime_hours', y='recommendation_pct', color='teal', scatter_kws={'s': 60})
plt.title('Median Playtime (Hours) vs Recommendation Rate (%)')
plt.xlabel('Median Playtime (Hours)')
plt.ylabel('Recommendation Rate (%)')
plt.tight_layout()
plt.show()

df_engagement.head(10)


### 4. Summary & Findings Checkpoint
- **Genre Dominance**: Action, Adventure, and RPG dominate both catalog counts and review counts.
- **Rank Divergence**: Identified multiple games with rank divergence (>30 rank gap between Sales Rank and Review Rank).
- **Reproducible SQL Queries**: Saved in `sql/business_queries.sql`.
